# Stage 08 / Lecture 56: Multi-Agent Agile SDLC Workflow with Human-in-the-Loop (HITL)

This reference architecture implements an end-to-end, multi-agent software engineering pipeline mimicking enterprise Agile software development:
1. **Product Manager / Business Analyst Agent (`PM_Agent`)**: Decomposes high-level client requirements into granular User Stories with Acceptance Criteria.
2. **Developer Agents (`Dev_Agent_1`, `Dev_Agent_2`)**: Produces typed Python implementations for allocated sprint tasks.
3. **Quality Assurance Agent (`QA_Agent`)**: Generates automated test suites, executes verification assertions, and raises structured defect tickets.
4. **Code Reviewer Agent (`Reviewer_Agent`)**: Analyzes code quality, algorithmic complexity, and security invariants.
5. **Human-in-the-Loop (HITL) Gatekeeper**: Confidence-thresholded escalation gating production deployments.


In [1]:
from typing import List, Dict, Optional, Tuple, Any, Callable
from dataclasses import dataclass, field
from enum import Enum
import re

print("Dependencies imported successfully.")


Dependencies imported successfully.


### 1. Domain Entities and SDLC State Models

We define formal dataclasses and status enumerations representing user stories, sprint backlogs, code review assessments, and quality reports.


In [2]:
class StoryStatus(str, Enum):
    BACKLOG = "BACKLOG"
    DEVELOPED = "DEVELOPED"
    REVIEW_PASSED = "REVIEW_PASSED"
    REVIEW_FAILED = "REVIEW_FAILED"
    QA_PASSED = "QA_PASSED"
    QA_FAILED = "QA_FAILED"
    DEPLOYED = "DEPLOYED"
    REJECTED_BY_HUMAN = "REJECTED_BY_HUMAN"


@dataclass
class UserStory:
    story_id: str
    title: str
    description: str
    assigned_dev: Optional[str] = None
    code_implementation: Optional[str] = None
    test_suite: Optional[str] = None
    review_notes: List[str] = field(default_factory=list)
    qa_defect_count: int = 0
    status: StoryStatus = StoryStatus.BACKLOG


@dataclass
class CodeReviewReport:
    story_id: str
    passed: bool
    quality_score: float
    violations: List[str]


@dataclass
class QATestReport:
    story_id: str
    passed: bool
    tests_executed: int
    defects_found: List[str]

print("Domain entities defined successfully.")


Domain entities defined successfully.


### 2. Multi-Agent Development Team Implementation

Each agent specializes in a distinct cognitive role within the software lifecycle, communicating via structured message contracts.


In [3]:
class ProductManagerAgent:
    """Decomposes raw client requirements into structured User Stories."""

    def plan_sprint(self, client_scope: str) -> List[UserStory]:
        stories = [
            UserStory(
                story_id="US-101",
                title="Revenue Calculation Engine",
                description="Implement compute_quarterly_growth(q1, q2, q3) returning percentage increase."
            ),
            UserStory(
                story_id="US-102",
                title="Input Sanitization Filter",
                description="Implement sanitize_financial_input(raw_val) validating non-negative floats."
            )
        ]
        return stories


class DeveloperAgent:
    """Simulates code generation for allocated sprint user stories."""

    def __init__(self, dev_id: str) -> None:
        self.dev_id = dev_id

    def implement(self, story: UserStory, bug_fix_context: Optional[str] = None) -> str:
        story.assigned_dev = self.dev_id
        if story.story_id == "US-101":
            if bug_fix_context:
                # Patched implementation fixing zero-division edge case
                code = (
                    "def compute_quarterly_growth(q1: float, q2: float, q3: float) -> float:\n"
                    "    if q1 <= 0:\n"
                    "        raise ValueError('Baseline revenue must be positive')\n"
                    "    return round(((q3 - q1) / q1) * 100.0, 2)"
                )
            else:
                # Buggy initial implementation (susceptible to q1 == 0)
                code = (
                    "def compute_quarterly_growth(q1: float, q2: float, q3: float) -> float:\n"
                    "    return ((q3 - q1) / q1) * 100.0"
                )
        else:
            code = (
                "def sanitize_financial_input(val: float) -> float:\n"
                "    if val < 0:\n"
                "        raise ValueError('Negative revenue invalid')\n"
                "    return float(val)"
            )
        story.code_implementation = code
        story.status = StoryStatus.DEVELOPED
        return code


class CodeReviewerAgent:
    """Performs static analysis, docstring audits, and security invariant verification."""

    def review(self, story: UserStory) -> CodeReviewReport:
        assert story.code_implementation is not None
        code = story.code_implementation
        violations = []

        if "-> float" not in code:
            violations.append("Missing return type annotation")
        if "raise ValueError" not in code and "compute_quarterly_growth" in code:
            violations.append("Lacks defensive exception handling for zero or invalid inputs")

        passed = len(violations) == 0
        score = 1.0 - (len(violations) * 0.3)
        story.status = StoryStatus.REVIEW_PASSED if passed else StoryStatus.REVIEW_FAILED
        story.review_notes.extend(violations)

        return CodeReviewReport(story_id=story.story_id, passed=passed, quality_score=score, violations=violations)


class QATestingAgent:
    """Generates test suites and executes dynamic assertion checks."""

    def test(self, story: UserStory) -> QATestReport:
        assert story.code_implementation is not None
        code = story.code_implementation
        scope: Dict[str, Any] = {}
        exec(code, scope)

        defects = []
        tests_executed = 0

        if story.story_id == "US-101":
            fn = scope["compute_quarterly_growth"]
            # Test 1: Standard growth
            tests_executed += 1
            if fn(100.0, 120.0, 150.0) != 50.0:
                defects.append("Failed standard growth calculation")
            # Test 2: Zero baseline division
            tests_executed += 1
            try:
                fn(0.0, 100.0, 150.0)
                defects.append("Zero baseline did not raise defensive ValueError")
            except ValueError:
                pass
            except ZeroDivisionError:
                defects.append("Zero baseline triggered unhandled ZeroDivisionError")

        passed = len(defects) == 0
        story.status = StoryStatus.QA_PASSED if passed else StoryStatus.QA_FAILED
        story.qa_defect_count = len(defects)
        return QATestReport(story_id=story.story_id, passed=passed, tests_executed=tests_executed, defects_found=defects)

print("Specialized agents initialized successfully.")


Specialized agents initialized successfully.


### 3. Human-in-the-Loop (HITL) Gatekeeper & Sprint Orchestrator

The enterprise orchestrator runs circular feedback between Developers, Reviewers, and Testers. If defects occur, the code is bounced back to the Developer. Upon passing all checks, it reaches the **Human-in-the-Loop** checkpoint before production deployment.


In [4]:
class EnterpriseAgileOrchestrator:
    """
    Coordinates the multi-agent software engineering lifecycle with
    circular feedback and Human-in-the-Loop (HITL) deployment gating.
    """

    def __init__(self, human_approval_callback: Callable[[UserStory], bool]) -> None:
        self.pm = ProductManagerAgent()
        self.dev = DeveloperAgent(dev_id="DevAgent_Alpha")
        self.reviewer = CodeReviewerAgent()
        self.qa = QATestingAgent()
        self.human_gate = human_approval_callback

    def execute_sprint_story(self, story: UserStory, max_repair_cycles: int = 3) -> UserStory:
        # 1. Initial Implementation
        self.dev.implement(story)

        # 2. Circular Feedback Loop (Review <-> QA <-> Repair)
        for cycle in range(max_repair_cycles):
            review_rep = self.reviewer.review(story)
            if not review_rep.passed:
                # Feedback loop: patch review issues
                self.dev.implement(story, bug_fix_context="; ".join(review_rep.violations))
                continue

            qa_rep = self.qa.test(story)
            if not qa_rep.passed:
                # Feedback loop: patch QA defects
                self.dev.implement(story, bug_fix_context="; ".join(qa_rep.defects))
                continue

            # Passed both static review and dynamic QA testing
            break

        assert story.status in (StoryStatus.QA_PASSED, StoryStatus.REVIEW_PASSED)

        # 3. Human-in-the-Loop Deployment Gate
        human_approved = self.human_gate(story)
        if human_approved:
            story.status = StoryStatus.DEPLOYED
        else:
            story.status = StoryStatus.REJECTED_BY_HUMAN

        return story


# Simulation Configuration with Human Approver
def mock_tech_lead_approval(story: UserStory) -> bool:
    # High-risk financial engines require explicit lead sign-off
    print(f"[HITL Gatekeeper] Reviewing {story.story_id}: '{story.title}'...")
    return story.qa_defect_count == 0


orchestrator = EnterpriseAgileOrchestrator(human_approval_callback=mock_tech_lead_approval)
sprint_backlog = orchestrator.pm.plan_sprint("Implement enterprise revenue analytics")

processed_stories = []
for story in sprint_backlog:
    finalized_story = orchestrator.execute_sprint_story(story)
    processed_stories.append(finalized_story)
    print(f"Story {finalized_story.story_id} Final Status: {finalized_story.status.value}")
    print(f"  Assigned Dev: {finalized_story.assigned_dev}")
    print(f"  Implementation:\n    {finalized_story.code_implementation.replace('\n', '\n    ')}")
    print("-" * 65)

# Verification Assertions
assert len(processed_stories) == 2
assert all(s.status == StoryStatus.DEPLOYED for s in processed_stories)
assert processed_stories[0].assigned_dev == "DevAgent_Alpha"
assert "raise ValueError" in processed_stories[0].code_implementation

print("All enterprise SDLC multi-agent assertions verified successfully.")


[HITL Gatekeeper] Reviewing US-101: 'Revenue Calculation Engine'...
Story US-101 Final Status: DEPLOYED
  Assigned Dev: DevAgent_Alpha
  Implementation:
    def compute_quarterly_growth(q1: float, q2: float, q3: float) -> float:
        if q1 <= 0:
            raise ValueError('Baseline revenue must be positive')
        return round(((q3 - q1) / q1) * 100.0, 2)
-----------------------------------------------------------------
[HITL Gatekeeper] Reviewing US-102: 'Input Sanitization Filter'...
Story US-102 Final Status: DEPLOYED
  Assigned Dev: DevAgent_Alpha
  Implementation:
    def sanitize_financial_input(val: float) -> float:
        if val < 0:
            raise ValueError('Negative revenue invalid')
        return float(val)
-----------------------------------------------------------------
All enterprise SDLC multi-agent assertions verified successfully.
